In [38]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd
import pandas as pd
import ast


In [39]:


# -------------------------------
# Input file
# -------------------------------
input_file = "results_deterministic_approximate.csv"

# -------------------------------
# Step 1: Robust manual parsing
# -------------------------------
rows = []

with open(input_file, "r") as f:
    for line_id, line in enumerate(f):
        line = line.strip()

        if not line:
            continue

        # Split ONLY first 3 commas
        parts = line.split(",", 3)

        if len(parts) < 4:
            print(f"Skipping malformed line {line_id}: {line}")
            continue

        file = parts[0]

        try:
            objective = float(parts[1])
            time = float(parts[2])
        except ValueError:
            print(f"Skipping numeric parse error at line {line_id}")
            continue

        solutions_raw = parts[3]

        # Convert string list → Python list
        try:
            solutions = ast.literal_eval(solutions_raw)
        except Exception:
            print(f"Warning: failed to parse solutions at line {line_id}")
            solutions = []

        rows.append([file, objective, time, solutions])

# -------------------------------
# Step 2: Create DataFrame
# -------------------------------
df = pd.DataFrame(rows, columns=["file", "objective", "time", "solutions"])

# -------------------------------
# Step 3: Filename parsing
# -------------------------------
def split_filename(fname):
    fname = fname.replace(".json", "")
    parts = fname.split("_")

    if len(parts) < 4:
        return pd.Series([fname, "", "", ""])

    if len(parts) > 4:
        instance_name = "_".join(parts[:-3])
        measure = parts[-3]
        percentile = parts[-2]
        algorithm = parts[-1]
    else:
        instance_name = parts[0]
        measure = parts[1]
        percentile = parts[2]
        algorithm = parts[3]

    return pd.Series([instance_name, measure, percentile, algorithm])

df[["file_name", "measure", "percentile", "algorithm"]] = df["file"].apply(split_filename)

# -------------------------------
# Step 4: Reorder columns
# -------------------------------
df = df[
    [
        "file_name",
        "measure",
        "percentile",
        "algorithm",
        "objective",
        "time",
        "solutions",
    ]
]

# -------------------------------
# Step 5: Add useful derived data
# -------------------------------
df["solution_size"] = df["solutions"].apply(len)

# Optional: density-like metric
df["solution_density"] = df["solution_size"] / df["objective"]

# -------------------------------
# Step 6: Memory optimization
# -------------------------------
df["objective"] = df["objective"].astype("int32")
df["time"] = df["time"].astype("float32")

# -------------------------------
# Step 7: Preview & sanity checks
# -------------------------------
df_greedy = df
df_greedy

Skipping numeric parse error at line 0


,file_name,measure,percentile,algorithm,objective,time,solutions,solution_size,solution_density
0,ela,cosine,p95,alg0,7805,25.053507,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",7805,1.0
1,transoptas_5pso,correlation,p80,alg1,5,1.834159,"[6564, 4325, 6565, 6566, 4397]",5,1.0
2,transoptas_5pso,cosine,p10,alg0,491,165.350204,"[4097, 8195, 4103, 4105, 4106, 4107, 4108, 410...",491,1.0
3,tinytla,seuclidean,p95,alg0,7463,19.015779,"[0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 15...",7463,1.0
4,transoptas_5pso,correlation,p20,alg0,963,98.866585,"[4097, 6146, 8195, 4100, 5, 2052, 4103, 4105, ...",963,1.0
...,...,...,...,...,...,...,...,...,...
175,ela,cosine,p80,alg1,13,3.561427,"[6752, 2341, 7464, 4937, 4939, 1548, 5070, 423...",13,1.0
176,transoptas_5pso,cosine,p80,alg0,5414,45.035755,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 17...",5414,1.0
177,deepela_large_r1,cosine,p50,alg1,2,3.385980,"[6718, 6575]",2,1.0
178,ela,cosine,p90,alg0,7476,33.878899,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",7476,1.0


In [40]:
import pandas as pd
import ast

# -------------------------------
# Input file
# -------------------------------
input_file = "results_exact.csv"

# -------------------------------
# Step 1: Robust manual parsing
# -------------------------------
rows = []

with open(input_file, "r") as f:
    for line_id, line in enumerate(f):
        line = line.strip()

        if not line:
            continue

        # Split ONLY first 3 commas
        parts = line.split(",", 3)

        if len(parts) < 4:
            print(f"Skipping malformed line {line_id}: {line}")
            continue

        file = parts[0]

        try:
            objective = float(parts[1])
            lb = float(parts[2])
        except ValueError:
            print(f"Skipping numeric parse error at line {line_id}")
            continue

        solution_raw = parts[3]

        # -------------------------------
        # Fix solution parsing (robust)
        # -------------------------------
        try:
            solution = ast.literal_eval(solution_raw)
        except Exception:
            # try removing quotes if double-encoded
            try:
                solution = ast.literal_eval(solution_raw.strip('"'))
            except Exception:
                print(f"Warning: failed to parse solution at line {line_id}")
                solution = []

        rows.append([file, objective, lb, solution])

# -------------------------------
# Step 2: Create DataFrame
# -------------------------------
df_exact = pd.DataFrame(rows, columns=["file", "objective", "lb", "solution"])

# -------------------------------
# Step 3: Filename parser
# -------------------------------
def split_filename(fname):
    fname = fname.replace(".json", "")
    parts = fname.split("_")

    if len(parts) < 4:
        return pd.Series([fname, "", "", ""])

    if len(parts) > 4:
        instance_name = "_".join(parts[:-3])
        measure = parts[-3]
        percentile = parts[-2]
        algorithm = parts[-1]
    else:
        instance_name = parts[0]
        measure = parts[1]
        percentile = parts[2]
        algorithm = parts[3]

    return pd.Series([instance_name, measure, percentile, algorithm])

df_exact[["file_name", "measure", "percentile", "algorithm"]] = df_exact["file"].apply(split_filename)

# -------------------------------
# Step 4: Reorder columns
# -------------------------------
df_exact = df_exact[
    [
        "file_name",
        "measure",
        "percentile",
        "algorithm",
        "objective",
        "lb",
        "solution",
    ]
]

# -------------------------------
# Step 5: Derived metrics
# -------------------------------
df_exact["solution_size"] = df_exact["solution"].apply(len)

# Gap (important for exact solvers)
df_exact["gap"] = df_exact["lb"] - df_exact["objective"]

# -------------------------------
# Step 6: Memory optimization
# -------------------------------
df_exact["objective"] = df_exact["objective"].astype("float32")
df_exact["lb"] = df_exact["lb"].astype("float32")

# -------------------------------
# Step 7: Debug / preview
# -------------------------------
#df_exact.head(10)



Skipping numeric parse error at line 0


In [67]:
import pandas as pd

# -------------------------------
# Step 1: Read CSV manually
# -------------------------------
input_file = "results_kmis_online_mis.csv"

data = []
with open(input_file, 'r') as f:
    header = f.readline()  # skip header
    for line in f:
        # split only on first and last comma
        file_part, rest = line.strip().split(',', 1)
        solution_part, objective_part = rest.rsplit(',', 1)
        data.append([file_part, solution_part, objective_part])

# -------------------------------
# Step 2: Create DataFrame
# -------------------------------
df = pd.DataFrame(data, columns=['file','solution','objective'])

# Convert objective to numeric
df['objective'] = pd.to_numeric(df['objective'], errors='coerce')

# -------------------------------
# Step 3: Parse filename
# -------------------------------
def parse_filename(fname):
    name = fname.replace(".out", "")
    parts = name.split("_")

    # Dataset name may have underscores
    dataset = "_".join(parts[:-3])
    measure = parts[-3]
    percentile = parts[-2]
    seed = parts[-1]

    return pd.Series([dataset, measure, percentile, seed])

df[['file_name','measure','percentile','seed']] = df['file'].apply(parse_filename)

# -------------------------------
# Step 4: Reorder columns
# -------------------------------
df = df[['file', 'file_name', 'measure', 'percentile', 'seed', 'solution', 'objective']]

# -------------------------------
# Step 5: Preview
# -------------------------------
df.head(30)

# -------------------------------
# Optional: Save cleaned CSV
# -------------------------------

## AGGREGATE:
df_online_kmis = df.drop(columns=["file"], axis=1)
df_online_kmis = df_online_kmis.sort_values(by=["file_name", "measure", "percentile", "seed"])
df_online_kmis[df_online_kmis["file_name"] == "doe2vec"]
#df_online_kmis.info()


,file_name,measure,percentile,seed,solution,objective
77,doe2vec,correlation,p10,10,"[33, 34, 35, 63, 64, 65, 78, 79, 80, 93, 94, ...",558
78,doe2vec,correlation,p10,20,"[33, 34, 35, 64, 65, 78, 79, 80, 93, 94, 95, ...",558
79,doe2vec,correlation,p10,30,"[33, 34, 35, 63, 64, 65, 78, 79, 80, 93, 94, ...",558
80,doe2vec,correlation,p10,40,"[33, 34, 35, 63, 64, 65, 78, 79, 80, 93, 94, ...",558
81,doe2vec,correlation,p10,50,"[33, 34, 35, 64, 65, 78, 79, 80, 93, 94, 95, ...",558
...,...,...,...,...,...,...
160,doe2vec,seuclidean,p95,10,"[1, 2, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17...",6583
161,doe2vec,seuclidean,p95,20,"[1, 2, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17...",6583
162,doe2vec,seuclidean,p95,30,"[1, 2, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17...",6584
163,doe2vec,seuclidean,p95,40,"[1, 2, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17...",6584


#REDU-MIS

In [42]:
import pandas as pd

# -------------------------------
# Step 1: Read CSV manually
# -------------------------------
input_file = "results_redumis.csv"

data = []
with open(input_file, 'r') as f:
    header = f.readline()  # skip header
    for line in f:
        # split only on first and last comma
        file_part, rest = line.strip().split(',', 1)
        solution_part, objective_part = rest.rsplit(',', 1)
        data.append([file_part, solution_part, objective_part])

# -------------------------------
# Step 2: Create DataFrame
# -------------------------------
df = pd.DataFrame(data, columns=['file','solution','objective'])

# Convert objective to numeric
df['objective'] = pd.to_numeric(df['objective'], errors='coerce')

# -------------------------------
# Step 3: Parse filename
# -------------------------------
def parse_filename(fname):
    name = fname.replace(".out", "")
    parts = name.split("_")

    # Dataset name may have underscores
    dataset = "_".join(parts[:-5])
    measure = parts[-5]
    percentile = parts[-4]
    seed = parts[-3][4:]

    return pd.Series([dataset, measure, percentile, seed])

df[['file_name','measure','percentile','seed']] = df['file'].apply(parse_filename)

# -------------------------------
# Step 4: Reorder columns
# -------------------------------
df = df[['file', 'file_name', 'measure', 'percentile', 'seed', 'solution', 'objective']]

# -------------------------------
# Step 5: Preview
# -------------------------------
df.head(30)

# -------------------------------
# Optional: Save cleaned CSV
# -------------------------------

## AGGREGATE:
df_redumis = df.drop(columns=["file"], axis=1).copy()

df_redumis

,file_name,measure,percentile,seed,solution,objective
0,deepela_large_r1,correlation,p10,30,"[45, 49, 51, 63, 93, 94, 108, 120, 121, 123, ...",505
1,deepela_large_r1,correlation,p10,40,"[45, 49, 51, 63, 93, 94, 108, 120, 121, 123, ...",505
2,deepela_large_r1,correlation,p10,50,"[45, 49, 51, 63, 93, 94, 108, 120, 121, 123, ...",505
3,deepela_large_r1,correlation,p20,20,"[45, 46, 49, 51, 60, 63, 72, 91, 93, 94, 97, ...",1055
4,deepela_large_r1,correlation,p20,50,"[45, 46, 49, 51, 60, 63, 72, 91, 93, 94, 97, ...",1055
...,...,...,...,...,...,...
310,transoptas_5pso,seuclidean,p90,50,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13...",6087
311,transoptas_5pso,seuclidean,p95,20,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13...",6790
312,transoptas_5pso,seuclidean,p95,30,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13...",6790
313,transoptas_5pso,seuclidean,p95,40,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13...",6790


#NUMDS

In [43]:
import pandas as pd
import ast  # for safely converting solution string to list

# -------------------------------
# Step 1: Read CSV safely
# -------------------------------
input_file = "results_numds.csv"

data = []
with open(input_file, 'r') as f:
    header = f.readline()  # skip header
    for line in f:
        line = line.strip()
        if not line:
            continue
        
        # Split only on the first three commas
        first_comma = line.find(',')
        second_comma = line.find(',', first_comma + 1)
        third_comma = line.find(',', second_comma + 1)
        
        name_part = line[:first_comma]
        seed_part = line[first_comma+1:second_comma]
        obj_part = line[second_comma+1:third_comma]
        sol_part = line[third_comma+1:]  # rest of line is solution
        
        # Convert numeric fields
        try:
            seed_val = int(seed_part)
            obj_val = int(obj_part)
        except ValueError:
            print(f"Skipping line (cannot parse numbers): {line}")
            continue
        
        data.append([name_part, seed_val, obj_val, sol_part])

# -------------------------------
# Step 2: Create DataFrame
# -------------------------------
df = pd.DataFrame(data, columns=['name','seed','objective','solution'])

# -------------------------------
# Step 3: Parse 'name' into dataset, measure, percentile
# -------------------------------
def parse_name(name):
    parts = name.split("_")
    if len(parts) < 3:
        return pd.Series([name, "", ""])
    dataset = "_".join(parts[:-2])
    measure = parts[-2]
    percentile = parts[-1]
    return pd.Series([dataset, measure, percentile])

df[['file_name','measure','percentile']] = df['name'].apply(parse_name)

# -------------------------------
# Step 4: Optional: Convert solution string to Python list
# -------------------------------
def parse_solution(sol_str):
    try:
        # safely convert string '[1,2,3]' to list
        return ast.literal_eval(sol_str)
    except:
        return []

df['solution_list'] = df['solution'].apply(parse_solution)

# -------------------------------
# Step 5: Reorder columns
# -------------------------------
df = df[['name','file_name','measure','percentile','seed','solution','objective']]

# -------------------------------
# Step 6: Preview
# --------------------------------

df_numds = df 
df_numds.drop(columns=["name"], axis=1, inplace=True)
df_numds = df_numds.sort_values(by=["file_name", "measure", "percentile", "seed"])
df_numds 
 

,file_name,measure,percentile,seed,solution,objective
72,deepela_large_r1,correlation,p10,10,[4359],1
74,deepela_large_r1,correlation,p10,20,[4359],1
129,deepela_large_r1,correlation,p10,30,[4359],1
319,deepela_large_r1,correlation,p10,40,[4359],1
194,deepela_large_r1,correlation,p10,50,[4359],1
...,...,...,...,...,...,...
249,transoptas_5pso,seuclidean,p95,10,"[556, 4335]",2
206,transoptas_5pso,seuclidean,p95,20,"[556, 4335]",2
335,transoptas_5pso,seuclidean,p95,30,"[556, 4335]",2
161,transoptas_5pso,seuclidean,p95,40,"[556, 4335]",2


In [44]:
#PREPARATION: df_greedy, df_exact, df_online_kmis, df_redumis, df_numds

df_approx = df_greedy[["file_name", "measure", "percentile", "algorithm", "solutions", "objective" ]]
df_approx_mis = df_approx[df_approx["algorithm"] == "alg0"]
df_approx_mds = df_approx[df_approx["algorithm"] == "alg1"]
#renaming

df_approx_mis.rename(columns={"objective" : "Greedy"}, inplace=True)
df_approx_mds.rename(columns={"objective" : "Greedy"}, inplace=True)
df_approx_mis.rename(columns={"solutions" : "solution_greedy"}, inplace=True)
df_approx_mds.rename(columns={"solutions" : "solution_greedy"}, inplace=True)

#df_approx_mis
#### EXACT SOLVERS

df_exact = df_exact[["file_name", "measure", "percentile", "algorithm", "objective", "lb", "solution" ]]
df_exact_mis = df_exact[df_exact["algorithm"] == "alg0"]
df_exact_mds = df_exact[df_exact["algorithm"] == "alg1"]

df_exact_mis.rename(columns={"objective" : "CPLEX"}, inplace=True) 
df_exact_mds.rename(columns={"objective" : "CPLEX"}, inplace=True) 
df_exact_mis.rename(columns={"solution" : "solution_exact"}, inplace=True) 
df_exact_mds.rename(columns={"solution" : "solution_exact"}, inplace=True)

df_approx_mis.drop(columns=["algorithm"], axis=1, inplace=True)
df_exact_mis.drop(columns=["algorithm"], axis=1, inplace=True)

df_approx_mds.drop(columns=["algorithm"], axis=1, inplace=True)
df_exact_mds.drop(columns=["algorithm"], axis=1, inplace=True)

### KMIS online:

#df_nclue = df_online_kmis[["instance_name", "measure", "percentile", "seed",  "objective" ]]
#df_nclue
df_online_kmis
df_online_kmis.rename(columns={"objective" : "objective_kmis"}, inplace=True)
df_online_kmis.rename(columns={"solution" : "solution_kmis"}, inplace=True)
df_online_kmis


/tmp/ipykernel_8571/1191139766.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_approx_mis.rename(columns={"objective" : "Greedy"}, inplace=True)
/tmp/ipykernel_8571/1191139766.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_approx_mds.rename(columns={"objective" : "Greedy"}, inplace=True)
/tmp/ipykernel_8571/1191139766.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_approx_mis.rename(columns={"solutions" : "soluti

,file_name,measure,percentile,seed,solution_kmis,objective_kmis
0,deepela_large_r1,correlation,p10,20,"[45, 49, 51, 63, 93, 94, 108, 120, 121, 123, ...",505
1,deepela_large_r1,correlation,p10,30,"[45, 49, 51, 63, 91, 93, 94, 106, 108, 120, 1...",505
2,deepela_large_r1,correlation,p10,50,"[45, 49, 51, 63, 93, 94, 108, 120, 121, 123, ...",504
3,deepela_large_r1,correlation,p20,10,"[45, 46, 49, 51, 60, 63, 72, 91, 93, 94, 97, ...",1055
4,deepela_large_r1,correlation,p20,20,"[45, 46, 49, 51, 60, 63, 72, 91, 93, 94, 97, ...",1055
...,...,...,...,...,...,...
306,tinytla,cosine,p95,10,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13...",7631
307,tinytla,cosine,p95,20,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13...",7631
308,tinytla,cosine,p95,30,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13...",7631
309,tinytla,cosine,p95,40,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13...",7631


In [45]:
## results_redumis.csv:
df_redumis = df_redumis[["file_name", "measure", "percentile", "seed", "solution", "objective" ]]
df_redumis

# Group by instance_name, measure, percentile, and compute mean objective
#df_redumis_grouped = df_redumis.groupby(["file_name", "measure", "percentile"], as_index=False).agg({
#    "objective": "mean"
#})

df_redumis.rename(columns={"objective" : "objective_redumis"}, inplace=True)
df_redumis.rename(columns={"solution" : "solution_redumis"}, inplace=True)

#df_redumis_grouped.drop(columns=["seed"], axis=1, inplace=True)
# Preview
#df_redumis_grouped.head(15)
df_redumis

,file_name,measure,percentile,seed,solution_redumis,objective_redumis
0,deepela_large_r1,correlation,p10,30,"[45, 49, 51, 63, 93, 94, 108, 120, 121, 123, ...",505
1,deepela_large_r1,correlation,p10,40,"[45, 49, 51, 63, 93, 94, 108, 120, 121, 123, ...",505
2,deepela_large_r1,correlation,p10,50,"[45, 49, 51, 63, 93, 94, 108, 120, 121, 123, ...",505
3,deepela_large_r1,correlation,p20,20,"[45, 46, 49, 51, 60, 63, 72, 91, 93, 94, 97, ...",1055
4,deepela_large_r1,correlation,p20,50,"[45, 46, 49, 51, 60, 63, 72, 91, 93, 94, 97, ...",1055
...,...,...,...,...,...,...
310,transoptas_5pso,seuclidean,p90,50,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13...",6087
311,transoptas_5pso,seuclidean,p95,20,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13...",6790
312,transoptas_5pso,seuclidean,p95,30,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13...",6790
313,transoptas_5pso,seuclidean,p95,40,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13...",6790


In [46]:
## numds
df_numds
df_numds = df_numds[["file_name", "measure", "percentile", "seed",  "objective", "solution" ]]


df_numds

# Group by instance_name, measure, percentile, and compute mean objective
#df_numds_grouped = df_numds.groupby(["file_name", "measure", "percentile"], as_index=False).agg({
#                "objective": "mean"
#})

df_numds.rename(columns={"solution" : "solution_numds" },   inplace=True)

In [47]:
## TO MERGE (MIS solvers):  

print("MIS solvers")
# List of dataframes to merge

def merge_and_output(dfs_to_merge = [df_online_kmis, df_redumis], columns_to_join = ["file_name", "measure", "percentile", "seed"], \
                     out_name="MIS_2mh_solvers_results.csv"):


    # Start with the first dataframe
    df_merged = dfs_to_merge[0]

    # Merge sequentially on the key columns
    for df in dfs_to_merge[1: ]:
        df_merged = df_merged.merge(df, on=columns_to_join, how="left")

    # Preview the merged dataframe  
    
    df_merged.to_csv(out_name)
    return df_merged

df_merged = merge_and_output()


MIS solvers


In [48]:
## MDS results to merge: df_approx_mds, df_exact_mds, df_numds...


## add solutions: 
df_merged_deterministic = merge_and_output(dfs_to_merge=[df_approx_mis, df_exact_mis], columns_to_join = ["file_name", "measure", "percentile"], \
                                          out_name="MIS_deterministic_solvers_results.csv")


################ MDS solvers -- results #######################

df_merged_deterministic = merge_and_output(dfs_to_merge=[df_approx_mds, df_exact_mds], columns_to_join = ["file_name", "measure", "percentile"], \
                                          out_name="MDS_deterministic_solvers_results.csv")

dh_mh = merge_and_output(dfs_to_merge=[df_numds], columns_to_join = ["file_name", "measure", "percentile"], \
                                          out_name="MDS_mh_solvers_results.csv") ## check the validity of solutions: remove those solutions non valid (returned due to encountered memory leak)


## SPLIT (70:30) RESULTS 


### read EXACT APPROACH FOR MIS AND MDS 


In [49]:
# ------------------ LOAD CSV ------------------
df = pd.read_csv("results_5train_test/results_deterministic_train_test_alg01.csv")

# ------------------ PARSE SOLUTION ------------------
# Convert string "[1,2,3]" → list
df["solution"] = df["solution"].apply(ast.literal_eval)

# ------------------ PARSE FILE NAME ------------------
def parse_filename(fname):
    # remove extension
    fname = fname.replace(".out", "")

    parts = fname.split("_")

    # last parts are always: percentile, alg, train/test (ela_correlation_p50_train_test_nodes_split_2_alg1_test.out)
    split_id = parts[-3] 
    percentile = parts[-8]
    algorithm = parts[-2]
    split = parts[-1]

    # measure is just before percentile
    measure = parts[-9]

    # everything before that is dataset name
    dataset = "_".join(parts[: (len(parts)-9)])   #parts[:-10])

    return pd.Series([dataset, measure, percentile, algorithm, split, split_id, ])


df[["dataset", "measure", "percentile", "algorithm", "split", "split_id"]] = df["file"].apply(parse_filename)

# ------------------ KEEP ONLY IMPORTANT COLUMNS ------------------
df = df[[
    "dataset",
    "measure",
    "percentile",
    "algorithm",
    "split",
    "split_id",
    "objective",
    "LB",
    "solution"
]]

# ------------------ DONE ------------------
df_MIS=df[df["algorithm"] == "alg0"]
df_MDS=df[df["algorithm"] == "alg1"]

In [50]:
# ------------------ LOAD CSV ------------------
df = pd.read_csv("results_5train_test/results_deterministic_train_test_alg23.csv")

# ------------------ PARSE SOLUTION ------------------
# Convert string "[1,2,3]" → list
df["solution"] = df["solution"].apply(ast.literal_eval)

# ------------------ PARSE FILE NAME ------------------
def parse_filename(fname):
    # remove extension
    fname = fname.replace(".out", "")

    parts = fname.split("_")

    # last parts are always: percentile, alg, train/test (ela_correlation_p50_train_test_nodes_split_2_alg1_test.out)
    split_id = parts[-3] 
    percentile = parts[-8]
    algorithm = parts[-2]
    split = parts[-1]

    # measure is just before percentile
    measure = parts[-9]

    # everything before that is dataset name
    dataset = "_".join(parts[: (len(parts)-9)])   #parts[:-10])

    return pd.Series([dataset, measure, percentile, algorithm, split, split_id, ])


df[["dataset", "measure", "percentile", "algorithm", "split", "split_id"]] = df["file"].apply(parse_filename)

# ------------------ KEEP ONLY IMPORTANT COLUMNS ------------------
df = df[[
    "dataset",
    "measure",
    "percentile",
    "algorithm",
    "split",
    "split_id",
    "objective",
    "solution"
]]

# ------------------ DONE ------------------
df_MIS_greedy=df[df["algorithm"] == "alg2"]
df_MDS_greedy=df[df["algorithm"] == "alg3"]

In [51]:
#df_MIS_greedy_sort = df_MIS_greedy.sort_values(
#    by=["dataset", "measure", "percentile", "split", "split_id"]
#)
#
#df_MIS_greedy_sort.head(20)

## kmis-online

In [52]:
# ------------------ PARSE SOLUTION -----------------


import pandas as pd
import ast

input_path = "results_5train_test/results_kmis_online_mis_5train_test.csv"


# ------------------ PARSE FILENAME ------------------
def parse_filename(fname):
    fname = fname.replace(".out", "")
    parts = fname.split("_")

    # ---- representation (everything before metric) ----
    metric_idx = next(i for i, p in enumerate(parts)
                      if p in ["correlation", "seuclidean", "cosine"])
    representation = "_".join(parts[:metric_idx])

    # ---- measure ----
    measure = parts[metric_idx]

    # ---- percentile ----
    percentile = next(p for p in parts if p.startswith("p"))

    # ---- split_id ----
    split_id = None
    if "split" in parts:
        split_idx = parts.index("split")
        split_id = int(parts[split_idx + 1])

    # ---- mode (train/test → last occurrence) ----
    mode = None
    for p in reversed(parts):
        if p in ["train", "test"]:
            mode = p
            break

    # ---- alg (important for THIS dataset) ----
    alg = None
    for p in parts:
        if p.startswith("alg"):
            alg = int(p.replace("alg", ""))

    return representation, measure, percentile, split_id, mode, alg


# ------------------ PARSE LINE ------------------
def parse_line(line):
    """
    Handles lines like:
    fname,222,2.54,222,"[1,2,3,...]"
    OR similar malformed CSV
    """

    # split first 4 commas → rest is solution
    parts = line.split(",", 4)

    fname = parts[0].strip()
    seed = int(parts[1].strip())
    time = float(parts[2].strip())      # runtime or score
    objective = int(parts[3].strip())

    # remove quotes if present
    sol_str = parts[4].strip().strip('"')

    solution = ast.literal_eval(sol_str)

    return fname, seed, time, objective, solution


# ------------------ MAIN ------------------
def main(input_path):
    rows = []

    with open(input_path, "r") as f:
        for line in f:
            line = line.strip()

            # skip header
            if not line or line.startswith("name"):
                continue

            try:
                fname, seed, time, objective, solution = parse_line(line)

                representation, measure, percentile, split_id, mode, alg = parse_filename(fname)

                rows.append({
                    "file": fname,
                    "representation": representation,
                    "measure": measure,
                    "percentile": percentile,
                    "split_id": split_id,
                    "mode": mode,
                    "alg": alg,
                    "seed": seed,
                    "time": time,
                    "objective": objective,
                    "solution_size": len(solution),
                    "solution": solution
                })

            except Exception as e:
                print(f"[ERROR] Failed parsing line:\n{line}\n{e}")

    df = pd.DataFrame(rows)
    return df


# ------------------ RUN ------------------
df_kmis_online = main(input_path)


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [53]:
df_kmis_small = df_kmis_online.drop(columns=["solution"])
df_kmis_small

KeyError: "['solution'] not found in axis"

## REDUMIS (results_redumis_5train_test.csv): TODO: convert node indices from solution (.txt file to be used)

In [54]:
input_path = "results_5train_test/results_redumis_5train_test.csv"

import os
import ast
import pandas as pd
import argparse


# ------------------ PARSE FILENAME ------------------
# ------------------ PARSE FILENAME ------------------
def parse_filename(fname):
    fname = fname.replace(".out", "")
    parts = fname.split("_")

    # ---- dataset (everything before metric) ----
    metric_idx = next(i for i, p in enumerate(parts) if p in ["correlation", "seuclidean", "cosine"])
    dataset = "_".join(parts[:metric_idx])

    # ---- measure ----
    measure = parts[metric_idx]

    # ---- percentile ----
    percentile = next(p for p in parts if p.startswith("p"))

    # ---- split_id ----
    split_id = None
    if "split" in parts:
        split_idx = parts.index("split")
        split_id = parts[split_idx + 1]

    # ---- train/test (last occurrence matters!) ----
    split = None
    for p in reversed(parts):
        if p in ["train", "test"]:
            split = p
            break

    # ---- seed ----
    seed = None
    for p in parts:
        if p.startswith("seed"):
            seed = int(p.replace("seed", ""))

    # ---- algorithm (last token, unless it's tXXX etc.) ----
    algorithm = parts[-1]

    return dataset, measure, percentile, split, split_id, seed, algorithm




# ------------------ PARSE LINE ------------------
def parse_line(line):
    # split only first and last comma safely
    first_comma = line.find(",")
    last_comma = line.rfind(",")

    fname = line[:first_comma].strip()
    solution_str = line[first_comma + 1:last_comma].strip()
    objective = int(line[last_comma + 1:].strip())

    solution = ast.literal_eval(solution_str)

    return fname, solution, objective


# ------------------ MAIN ------------------
def main(input_path):

    rows = []

    with open(input_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            try:
                fname, solution, objective = parse_line(line)

                dataset, measure, percentile, split, split_id, seed, algorithm = parse_filename(fname)

                rows.append({
                    "file": fname,
                    "dataset": dataset,
                    "measure": measure,
                    "percentile": percentile,
                    "split": split,
                    "split_id": split_id,
                    "seed": seed,
                    "algorithm": algorithm,
                    "objective": objective,
                    "solution_size": len(solution),
                    "solution": solution
                })

            except Exception as e:
                print(f"[ERROR] Failed parsing line:\n{line}\n{e}")

    df = pd.DataFrame(rows)
    return df
    # save
    #df.to_csv(args.output_csv, index=False)

    #print(f"\nSaved to: {args.output_csv}")


if __name__ == "__main__":
    df_redumis = main(input_path)
    

In [60]:
#df_redumis.drop(columns=["file"], axis=1, inplace=True)

df_redumis = df_redumis.sort_values(by=["dataset", "measure", "percentile", "split", "split_id", "seed"])
df_redumis

,dataset,measure,percentile,split,split_id,seed,algorithm,objective,solution_size,solution
0,doe2vec,correlation,p10,test,1,10,redumis,172,172,"[21, 29, 36, 53, 58, 59, 65, 72, 81, 82, 83, 8..."
1,doe2vec,correlation,p10,test,1,20,redumis,172,172,"[21, 29, 36, 53, 58, 59, 64, 65, 72, 81, 82, 8..."
2,doe2vec,correlation,p10,test,1,30,redumis,172,172,"[21, 29, 36, 53, 58, 59, 65, 72, 81, 82, 83, 8..."
3,doe2vec,correlation,p10,test,1,40,redumis,172,172,"[21, 29, 36, 53, 58, 59, 64, 65, 72, 81, 82, 8..."
4,doe2vec,correlation,p10,test,1,50,redumis,172,172,"[21, 29, 36, 53, 58, 59, 65, 72, 81, 82, 83, 8..."
...,...,...,...,...,...,...,...,...,...,...
2909,transoptas_5pso,seuclidean,p95,train,5,10,redumis,4709,4709,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14,..."
2910,transoptas_5pso,seuclidean,p95,train,5,20,redumis,4709,4709,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14,..."
2911,transoptas_5pso,seuclidean,p95,train,5,30,redumis,4709,4709,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14,..."
2912,transoptas_5pso,seuclidean,p95,train,5,40,redumis,4709,4709,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14,..."


In [56]:
## kmis-online 

In [69]:
input_path = "results_5train_test/results_kmis_online_mis_5train_test.csv"



# ------------------ PARSE FILENAME ------------------
def parse_filename(fname):
    fname = fname.replace(".out", "")
    parts = fname.split("_")

    # ---- dataset (everything before metric) ----
    metric_idx = next(i for i, p in enumerate(parts) if p in ["correlation", "seuclidean", "cosine"])
    dataset = "_".join(parts[:metric_idx])

    # ---- measure ----
    measure = parts[metric_idx]

    # ---- percentile ----
    percentile = next(p for p in parts if p.startswith("p"))

    # ---- split_id ----
    split_id = None
    if "split" in parts:
        split_idx = parts.index("split")
        split_id = parts[split_idx + 1]

    # ---- train/test (last occurrence matters!) ----
    split = None
    for p in reversed(parts):
        if p in ["train", "test"]:
            split = p
            break

    # ---- seed ----
    seed = None
    for p in parts:
        if p.startswith("seed"):
            seed = int(p.replace("seed", ""))

    # ---- algorithm (last token, unless it's tXXX etc.) ----
    algorithm = parts[-1]

    return dataset, measure, percentile, split, split_id, seed, algorithm



# ------------------ PARSE LINE ------------------
def parse_line(line):
    # split only first and last comma safely
    first_comma = line.find(",")
    last_comma = line.rfind(",")

    fname = line[:first_comma].strip()
    solution_str = line[first_comma + 1:last_comma].strip()
    objective = int(line[last_comma + 1:].strip())

    solution = ast.literal_eval(solution_str)

    return fname, solution, objective


# ------------------ MAIN ------------------
def main(input_path):

    rows = []

    with open(input_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            try:
                fname, solution, objective = parse_line(line)

                dataset, measure, percentile, split, split_id, seed, algorithm = parse_filename(fname)

                rows.append({
                    "file": fname,
                    "dataset": dataset,
                    "measure": measure,
                    "percentile": percentile,
                    "split": split,
                    "split_id": split_id,
                    "seed": seed,
                    "algorithm": algorithm,
                    "objective": objective,
                    "solution_size": len(solution),
                    "solution": solution
                })

            except Exception as e:
                print(f"[ERROR] Failed parsing line:\n{line}\n{e}")

    df = pd.DataFrame(rows)
    return df
    # save
    #df.to_csv(args.output_csv, index=False)

    #print(f"\nSaved to: {args.output_csv}")


if __name__ == "__main__":
    df_kmis = main(input_path)
    

In [70]:
df_kmis.head(20)

,file,dataset,measure,percentile,split,split_id,seed,algorithm,objective,solution_size,solution
0,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,test,1,10,redumis,172,172,"[21, 29, 36, 53, 58, 59, 65, 72, 81, 82, 83, 8..."
1,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,test,1,20,redumis,172,172,"[21, 29, 36, 53, 58, 59, 64, 65, 72, 81, 82, 8..."
2,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,test,1,30,redumis,172,172,"[21, 29, 36, 53, 58, 59, 65, 72, 81, 82, 83, 8..."
3,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,test,1,40,redumis,172,172,"[21, 29, 36, 53, 58, 59, 65, 72, 81, 82, 83, 8..."
4,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,test,1,50,redumis,172,172,"[21, 29, 36, 53, 58, 59, 65, 72, 81, 82, 83, 8..."
5,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,train,1,10,redumis,403,403,"[21, 22, 23, 42, 49, 50, 57, 58, 66, 67, 68, 7..."
6,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,train,1,20,redumis,403,403,"[21, 22, 23, 49, 50, 57, 58, 66, 67, 68, 77, 7..."
7,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,train,1,30,redumis,403,403,"[21, 22, 23, 49, 50, 57, 58, 66, 67, 68, 77, 7..."
8,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,train,1,40,redumis,403,403,"[21, 22, 23, 49, 50, 57, 58, 66, 67, 68, 77, 7..."
9,doe2vec_correlation_p10_train_test_nodes_split...,doe2vec,correlation,p10,train,1,50,redumis,403,403,"[21, 22, 23, 49, 50, 57, 58, 66, 67, 68, 77, 7..."


## NuMDS

In [6]:
input_path = "results_5train_test/results_numds_5train_test.csv"

import pandas as pd
import ast

rows = []

with open(input_path, "r") as f:
    next(f)  # skip header
    
    for line in f:
        line = line.strip()
        if not line:
            continue
        
        # Fix CSV issue (list column)
        parts = line.split(",", 3)
        
        name = parts[0]
        seed = int(parts[1])
        obj = int(parts[2])
        sol = ast.literal_eval(parts[3])
        
        # ---- Parse name ----
        tokens = name.split("_")
        
        # Find key positions
        metric_idx = next(i for i, t in enumerate(tokens) if t in ["correlation", "seuclidean", "cosine"])
        p_idx = next(i for i, t in enumerate(tokens) if t.startswith("p"))
        split_idx = tokens.index("split")
        
        representation = "_".join(tokens[:metric_idx])
        metric = tokens[metric_idx]
        percentile = tokens[p_idx]
        nodes_split = tokens[split_idx + 1]# int(tokens[split_idx + 1])
        mode = tokens[-1]  # train/test
        
        rows.append({
            "name": name,
            "representation": representation,
            "metric": metric,
            "percentile": percentile,
            "nodes_split": nodes_split,
            "mode": mode,
            "seed": seed,
            "obj": obj,
            "sol": sol
        })

df = pd.DataFrame(rows)
dfpercentile

,name,representation,metric,percentile,nodes_split,mode,seed,obj,sol
0,transoptas_5pso_correlation_p95_train_test_nod...,transoptas_5pso,correlation,p95,1,train,10,8,"[4618, 3026, 4297, 151, 1233, 4761, 2972, 3113]"
1,tinytla_seuclidean_p80_train_test_nodes_split_...,tinytla,seuclidean,p80,5,test,20,1,[74]
2,tinytla_seuclidean_p95_train_test_nodes_split_...,tinytla,seuclidean,p95,2,train,20,1,[6]
3,tinytla_seuclidean_p90_train_test_nodes_split_...,tinytla,seuclidean,p90,4,train,20,1,[4]
4,transoptas_5pso_correlation_p95_train_test_nod...,transoptas_5pso,correlation,p95,3,test,40,25,"[1287, 2059, 2079, 1837, 52, 109, 1508, 666, 2..."
...,...,...,...,...,...,...,...,...,...
918,deepela_large_r1_seuclidean_p90_train_test_nod...,deepela_large_r1,seuclidean,p90,5,train,20,1,[2385]
919,doe2vec_seuclidean_p90_train_test_nodes_split_...,doe2vec,seuclidean,p90,5,test,20,2,"[452, 1334]"
920,tinytla_seuclidean_p95_train_test_nodes_split_...,tinytla,seuclidean,p95,5,train,50,1,[4]
921,ela_cosine_p90_train_test_nodes_split_5_train,ela,cosine,p90,5,train,20,5,"[1976, 1743, 1981, 3538, 3612]"
